# Phase 1: Phenotype Definition — HL Cases and Controls in PMBB

Docs in: https://halllab.atlassian.net/wiki/spaces/HLP/pages/edit-v2/730759171?draftShareId=1b5539e5-acee-41c7-a45f-71a40f75b083

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os

In [ ]:
# Detect current notebook directory
# ramp up until we find the project root (which contains 'data/raw')
notebook_dir = Path().resolve()
print(f"Diretório atual: {notebook_dir}")

# Search for project root by hierarchy
project_root = notebook_dir
for _ in range(5):  # sobe até 5 níveis
    if (project_root / "data" / "pmbb_v3").exists():
        break
    project_root = project_root.parent

print(f"Project root found: {project_root}")
os.chdir(project_root)
print(f"Working directory fixed to: {Path().resolve()}")

# Paths
PHECODE12  = project_root / "data/raw/phecode12_long.txt"
PHECODEX   = project_root / "data/raw/phecodeX_long.txt"
COVARIATES = project_root / "data/raw/covariates.txt"

# Check if files exist
for name, path in [("PHECODE12", PHECODE12), ("PHECODEX", PHECODEX), ("COVARIATES", COVARIATES)]:
    status = "✅" if path.exists() else "❌"
    print(f"{status} {name}: {path}")

print(f"\nPandas version: {pd.__version__}")

Diretório atual: /project/hall/analysis/hearing-loss-genomics/notebooks/01_phase1_exploration
Project root encontrado: /project/hall/analysis/hearing-loss-genomics
Working directory fixado para: /project/hall/analysis/hearing-loss-genomics
✅ PHECODE12: /project/hall/analysis/hearing-loss-genomics/data/raw/phecode12_long.txt
✅ PHECODEX: /project/hall/analysis/hearing-loss-genomics/data/raw/phecodeX_long.txt
✅ COVARIATES: /project/hall/analysis/hearing-loss-genomics/data/raw/covariates.txt

Pandas version: 3.0.2


---------------

## Phecode 389 — Results comparison with reference paper

### Context
Hui et al. 2023 used the PMBB with ~40K exomes and defined cases as participants
with **phecode 389 exact** and 2 or more EHR occurrences.
This analysis uses **PMBB Release 2024-3.0** with ~55K participants.

Two phenotyping strategies were tested:
- **Strategy A — Phecode 389 exact:** directly replicates the paper methodology
- **Strategy B — Phecode 389 family (389.x):** includes all subtypes (conductive HL,
  sensorineural HL, unspecified HL), capturing a broader hearing loss population

### Results

| Group | Hui et al. 2023 (~40K) | Strategy A — 389 exact | Strategy B — 389 family |
|---|---|---|---|
| Candidate cases (num_visits ≥ 2) | ~3,304 | 2,797 | 8,816 |
| Excluded / NA (num_visits = 1) | — | 2,680 | 6,994 |
| Potential controls (no phecode 389) | ~34,704 | 50,342 | 46,840 |

### Observations

> ⚠️ **Strategy A (389 exact):** Case count (2,797) is lower than published (3,304)
> despite the larger cohort. Possible explanations: (1) differences in phecode
> mapping between PMBB releases; (2) changes in ICD→phecode conversion between
> PheCode versions. This strategy most faithfully replicates the original methodology.

> 📈 **Strategy B (389 family):** Substantially increases case count (8,816 vs 2,797)
> by capturing all HL subtypes. More clinically inclusive but diverges from the
> original paper approach.

### Pending decisions
- [  ] Confirm which PheCode version was used in the original ZNF175 study (unpublished)
- [  ] Align with Nikki Palmiero / Molly Hall on which strategy to adopt
- [  ] Recommended: run both strategies in parallel and compare gene burden results

### Analysis: Strategy A — 389 exact

In [21]:
# Peek at first 5 rows without loading full file
df_peek = pd.read_csv(PHECODE12, sep="\t", nrows=5)
print("Shape preview (5 rows):", df_peek.shape)
print("\nColumn names:")
print(df_peek.columns.tolist())
print("\nFirst rows:")
df_peek

Shape preview (5 rows): (5, 5)

Column names:
['person_id', 'Phecode', 'num_visits', 'first_date', 'last_date']

First rows:


,person_id,Phecode,num_visits,first_date,last_date
0,PMBB1000274307312,208.00,2,2020-10-05,2020-10-05
1,PMBB1000274307312,211.00,3,2013-11-27,2017-04-30
2,PMBB1000274307312,244.20,7,2016-01-10,2022-09-17
3,PMBB1000274307312,244.40,25,2005-06-04,2020-07-26
4,PMBB1000274307312,250.42,15,2012-05-27,2021-03-02


In [22]:
df = pd.read_csv(PHECODE12, sep="\t", 
                 usecols=["person_id", "Phecode", "num_visits"],
                 dtype={"person_id": str, "Phecode": str})

# Filter phecode 389
hl_389 = df[df["Phecode"] == "389"]

print(f"Total participants with phecode 389: {hl_389['person_id'].nunique():,}")
print(f"Total lines:                        {len(hl_389):,}")
print(f"\nDistribution of num_visits:")
print(hl_389["num_visits"].describe())
print(f"\nFirst rows:")
hl_389.head(10)

Total participants with phecode 389: 5,477
Total lines:                        5,477

Distribution of num_visits:
count    5477.000000
mean        2.877670
std         4.073513
min         1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max        75.000000
Name: num_visits, dtype: float64

First rows:


,person_id,Phecode,num_visits
712,PMBB1002543221014,389,1
1222,PMBB1003773408607,389,1
1881,PMBB1005955578248,389,2
2173,PMBB1006628417880,389,1
4070,PMBB1014193377925,389,2
4359,PMBB1014648509276,389,1
4892,PMBB1016122029769,389,5
5017,PMBB1016925843753,389,1
5354,PMBB1017704026345,389,3
6213,PMBB1020468613439,389,1


In [23]:
# Classif directyly by num_visits
cases    = hl_389[hl_389["num_visits"] >= 2]["person_id"]
excluded = hl_389[hl_389["num_visits"] == 1]["person_id"]

# Controles = ID without phecode 389
all_ids  = pd.Series(df["person_id"].unique())
controls = all_ids[~all_ids.isin(hl_389["person_id"])]

print("=== Phase 1 — Phecode 389 Classification ===")
print(f"Candidate cases   (num_visits >= 2): {len(cases):>8,}")
print(f"Excluded / NA     (num_visits == 1): {len(excluded):>8,}")
print(f"Potential controls (no phecode 389): {len(controls):>8,}")

=== Phase 1 — Phecode 389 Classification ===
Candidate cases   (num_visits >= 2):    2,797
Excluded / NA     (num_visits == 1):    2,680
Potential controls (no phecode 389):   50,342


### Analysis: Strategy B — 389 family

In [24]:
hl_family = df[df["Phecode"].astype(str).str.startswith("389", na=False)]

In [25]:
print(f"Total participants with phecode 389: {hl_family['person_id'].nunique():,}")
print(f"Total lines:                        {len(hl_family):,}")
print(f"\nDistribution of num_visits:")
print(hl_family["num_visits"].describe())
print(f"\nFirst rows:")
hl_family.head(10)

Total participants with phecode 389: 8,979
Total lines:                        15,810

Distribution of num_visits:
count    15810.000000
mean         3.238710
std          4.701617
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max        138.000000
Name: num_visits, dtype: float64

First rows:


,person_id,Phecode,num_visits
712,PMBB1002543221014,389,1
713,PMBB1002543221014,389.4,2
1037,PMBB1003454137566,389.1,5
1038,PMBB1003454137566,389.5,1
1222,PMBB1003773408607,389,1
1881,PMBB1005955578248,389,2
1882,PMBB1005955578248,389.1,4
2043,PMBB1006264295351,389.4,1
2173,PMBB1006628417880,389,1
2174,PMBB1006628417880,389.1,21


In [26]:
# Classif directyly by num_visits, but only for phecode 389 family (389, 389.1, 389.2, etc.)
cases_family    = hl_family[hl_family["num_visits"] >= 2]["person_id"]
excluded_family = hl_family[hl_family["num_visits"] == 1]["person_id"]

# Controls = All unique person_ids in df that are NOT in hl_family
all_ids_family  = pd.Series(df["person_id"].unique())
controls_family = all_ids_family[~all_ids_family.isin(hl_family["person_id"])]

print("=== Phase 1 — Phecode 389 Classification ===")
print(f"Candidate cases   (num_visits >= 2): {len(cases_family):>8,}")
print(f"Excluded / NA     (num_visits == 1): {len(excluded_family):>8,}")
print(f"Potential controls (no phecode 389): {len(controls_family):>8,}")

=== Phase 1 — Phecode 389 Classification ===
Candidate cases   (num_visits >= 2):    8,816
Excluded / NA     (num_visits == 1):    6,994
Potential controls (no phecode 389):   46,840


---------------------

Test with PHECODEX instead PHECODE12. But Phecodes are differents!

In [27]:
df_x = pd.read_csv(PHECODEX, sep="\t", 
                 usecols=["person_id", "Phecode", "num_visits"],
                 dtype={"person_id": str, "Phecode": str})

df_x.head(10)


,person_id,Phecode,num_visits
0,PMBB1000274307312,CA_136,5
1,PMBB1000274307312,CA_136.2,1
2,PMBB1000274307312,CA_136.4,2
3,PMBB1000274307312,CA_136.41,2
4,PMBB1000274307312,CV_401,46
5,PMBB1000274307312,CV_401.1,46
6,PMBB1000274307312,CV_439,2
7,PMBB1000274307312,DE_662,10
8,PMBB1000274307312,DE_679,1
9,PMBB1000274307312,DE_679.1,1


----------

## Analise de Covariantes

In [18]:
cov = pd.read_csv(COVARIATES, sep="\t")

print("=== Covariates File ===")
print(f"Shape: {cov.shape}")
print(f"\nColumns: {cov.columns.tolist()}")
print(f"\nData types:\n{cov.dtypes}")
print(f"\nFirst rows:")
cov.head()

=== Covariates File ===
Shape: (57170, 13)

Columns: ['person_id', 'CREP_HighRisk_Flag', 'Sample_age', 'Sample_date', 'Batch', 'Sequenced_gender', 'Class', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6']

Data types:
person_id                 str
CREP_HighRisk_Flag      int64
Sample_age            float64
Sample_date               str
Batch                   int64
Sequenced_gender          str
Class                     str
PC1                   float64
PC2                   float64
PC3                   float64
PC4                   float64
PC5                   float64
PC6                   float64
dtype: object

First rows:


,person_id,CREP_HighRisk_Flag,Sample_age,Sample_date,Batch,Sequenced_gender,Class,PC1,PC2,PC3,PC4,PC5,PC6
0,PMBB1000274307312,0,48.0,2014-12-14,1,Male,EUR,0.011656,0.042763,-0.013940,-0.013713,-0.002553,-0.008277
1,PMBB1000437739273,0,53.5,2015-12-28,1,Female,EUR,0.013396,0.050600,-0.014085,-0.003402,0.000191,-0.011688
2,PMBB1000614770665,0,74.9,2017-05-01,2,Male,EUR,0.012050,0.045067,-0.017490,-0.009806,-0.000195,0.002982
3,PMBB1000856639250,0,72.6,2017-10-18,1,Female,EUR,0.015110,0.050735,-0.027207,-0.000464,0.003725,-0.002348
4,PMBB1001117453706,0,37.5,2016-03-25,1,Female,EUR,0.012419,0.049415,-0.018753,-0.006879,0.013544,-0.001271


In [19]:
print("=== Open Questions Resolution ===\n")

# Q1: PC count
pc_cols = [c for c in cov.columns if c.upper().startswith("PC")]
print(f"Q1 - Number of PCs available: {len(pc_cols)}")
print(f"     PC columns: {pc_cols}")

# Q2: Age field
age_cols = [c for c in cov.columns if "age" in c.lower() or "birth" in c.lower() or "date" in c.lower() or "enroll" in c.lower()]
print(f"\nQ2 - Age-related columns: {age_cols}")

# Q3: Sex field
sex_cols = [c for c in cov.columns if "sex" in c.lower() or "gender" in c.lower()]
print(f"\nQ3 - Sex column: {sex_cols}")
print(f"\nSex distribution:")
if sex_cols:
    print(cov[sex_cols[0]].value_counts())

=== Open Questions Resolution ===

Q1 - Number of PCs available: 6
     PC columns: ['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6']

Q2 - Age-related columns: ['Sample_age', 'Sample_date']

Q3 - Sex column: ['Sequenced_gender']

Sex distribution:
Sequenced_gender
Female    29452
Male      27718
Name: count, dtype: int64


In [20]:
# === Investigar Class e CREP_HighRisk_Flag ===

# Distribuição de ancestralidade (Class)
print("=== Ancestry distribution (Class) ===")
print(cov["Class"].value_counts())
print(f"Total: {len(cov):,}")

# CREP_HighRisk_Flag
print("\n=== CREP_HighRisk_Flag distribution ===")
print(cov["CREP_HighRisk_Flag"].value_counts())

# Quantos sobram após filtrar apenas EUR e AFR (como no paper)
eur_afr = cov[cov["Class"].isin(["EUR", "AFR"])]
print(f"\n=== After filtering EUR + AFR only ===")
print(f"Participants remaining: {len(eur_afr):,}")
print(f"Removed: {len(cov) - len(eur_afr):,}")

# Sample_age distribution
print("\n=== Sample_age distribution ===")
print(cov["Sample_age"].describe())
print(f"Missing ages: {cov['Sample_age'].isna().sum():,}")

=== Ancestry distribution (Class) ===
Class
EUR         42133
AFR         12113
EAS           907
AMR           761
SAS           743
UNKNOWN0      448
UNKNOWN1       59
UNKNOWN2        6
Name: count, dtype: int64
Total: 57,170

=== CREP_HighRisk_Flag distribution ===
CREP_HighRisk_Flag
0    54337
1     2833
Name: count, dtype: int64

=== After filtering EUR + AFR only ===
Participants remaining: 54,246
Removed: 2,924

=== Sample_age distribution ===
count    57170.000000
mean        54.272162
std         16.530732
min         18.000000
25%         41.400000
50%         56.300000
75%         66.700000
max         90.000000
Name: Sample_age, dtype: float64
Missing ages: 0


In [29]:
# ============================================================
# CELL: Full pipeline — load, filter, classify, apply QC
# Run this cell to restore all variables
# ============================================================
from pathlib import Path
import pandas as pd

# --- Paths ---
project_root = Path("/project/hall/analysis/hearing-loss-genomics")
PHECODE12  = project_root / "data/raw/phecode12_long.txt"
COVARIATES = project_root / "data/raw/covariates.txt"

# --- Load phecodes ---
print("Loading phecodes...")
df = pd.read_csv(PHECODE12, sep="\t",
                 usecols=["person_id", "Phecode", "num_visits"],
                 dtype={"person_id": str,
                        "Phecode": "category",
                        "num_visits": "float32"})
print(f"  Phecode file: {len(df):,} rows, {df['person_id'].nunique():,} participants")

# --- Load covariates ---
print("Loading covariates...")
cov = pd.read_csv(COVARIATES, sep="\t")
print(f"  Covariates file: {len(cov):,} participants")

# --- Strategy A: phecode 389 exact (no QC) ---
hl_exact   = df[df["Phecode"].astype(str) == "389"]
cases      = hl_exact[hl_exact["num_visits"] >= 2]["person_id"]
excluded   = hl_exact[hl_exact["num_visits"] == 1]["person_id"]
all_ids    = pd.Series(df["person_id"].unique())
controls   = all_ids[~all_ids.isin(hl_exact["person_id"])]

# --- Strategy B: phecode 389 family (no QC) ---
hl_family     = df[df["Phecode"].astype(str).str.startswith("389", na=False)]
hl_family_agg = hl_family.groupby("person_id")["num_visits"].max().reset_index()
cases_fam     = hl_family_agg[hl_family_agg["num_visits"] >= 2]["person_id"]
excluded_fam  = hl_family_agg[hl_family_agg["num_visits"] == 1]["person_id"]
controls_fam  = all_ids[~all_ids.isin(hl_family_agg["person_id"])]

# --- QC filters (replicating paper) ---
print("Applying QC filters...")
cov_qc  = cov[
    (cov["CREP_HighRisk_Flag"] == 0) &
    (cov["Class"].isin(["EUR", "AFR"])) &
    (cov["Sample_age"].notna())
]
qc_ids  = set(cov_qc["person_id"])

# --- Strategy A + QC ---
hl_exact_qc   = hl_exact[hl_exact["person_id"].isin(qc_ids)]
cases_qc      = hl_exact_qc[hl_exact_qc["num_visits"] >= 2]["person_id"]
excluded_qc   = hl_exact_qc[hl_exact_qc["num_visits"] == 1]["person_id"]
all_qc_ids    = pd.Series(list(qc_ids))
controls_qc   = all_qc_ids[~all_qc_ids.isin(hl_exact_qc["person_id"])]

# --- Summary ---
print("\n========== SUMMARY ==========")
print(f"\nParticipants after QC filters:  {len(cov_qc):>8,}  (paper: 36,507)")
print(f"\nStrategy A — phecode 389 exact (no QC):")
print(f"  Cases     (>= 2): {len(cases):>8,}  | Paper: ~3,304")
print(f"  Excluded  (== 1): {len(excluded):>8,}")
print(f"  Controls:         {len(controls):>8,}  | Paper: ~34,704")
print(f"\nStrategy B — phecode 389 family (no QC):")
print(f"  Cases     (>= 2): {len(cases_fam):>8,}")
print(f"  Excluded  (== 1): {len(excluded_fam):>8,}")
print(f"  Controls:         {len(controls_fam):>8,}")
print(f"\nStrategy A + QC filters (EUR+AFR, CREP=0):")
print(f"  Cases     (>= 2): {len(cases_qc):>8,}  | Paper: ~3,304")
print(f"  Excluded  (== 1): {len(excluded_qc):>8,}")
print(f"  Controls:         {len(controls_qc):>8,}  | Paper: ~34,704")
print("\nAll variables restored successfully.")

Loading phecodes...
  Phecode file: 2,882,965 rows, 55,819 participants
Loading covariates...
  Covariates file: 57,170 participants
Applying QC filters...

========== SUMMARY ==========

Participants after QC filters:    51,506  (paper: 36,507)

Strategy A — phecode 389 exact (no QC):
  Cases     (>= 2):    2,797  | Paper: ~3,304
  Excluded  (== 1):    2,680
  Controls:           50,342  | Paper: ~34,704

Strategy B — phecode 389 family (no QC):
  Cases     (>= 2):    5,664
  Excluded  (== 1):    3,315
  Controls:           46,840

Strategy A + QC filters (EUR+AFR, CREP=0):
  Cases     (>= 2):    2,641  | Paper: ~3,304
  Excluded  (== 1):    2,501
  Controls:           46,364  | Paper: ~34,704

All variables restored successfully.


In [30]:
# Investigar os "excluídos" (num_visits == 1) para entender a diferença
print("=== Excluded participants (num_visits == 1) ===")
hl_excluded = hl_exact[hl_exact["num_visits"] == 1]

print(f"Total excluded: {len(hl_excluded):,}")
print(f"\nfirst_date vs last_date — same date? (single visit)")

# Precisamos recarregar com as datas para investigar
df_with_dates = pd.read_csv(PHECODE12, sep="\t",
                             usecols=["person_id", "Phecode", "num_visits",
                                      "first_date", "last_date"],
                             dtype={"person_id": str, "Phecode": "category",
                                    "num_visits": "float32"})

hl_excl_dates = df_with_dates[
    (df_with_dates["Phecode"].astype(str) == "389") &
    (df_with_dates["num_visits"] == 1)
]

# Verificar se first_date == last_date (genuinamente 1 visita)
same_date = (hl_excl_dates["first_date"] == hl_excl_dates["last_date"]).sum()
diff_date = (hl_excl_dates["first_date"] != hl_excl_dates["last_date"]).sum()

print(f"first_date == last_date (true single visit): {same_date:,}")
print(f"first_date != last_date (different visits!): {diff_date:,}")
print(f"\nSample of excluded with different dates:")
print(hl_excl_dates[hl_excl_dates["first_date"] != hl_excl_dates["last_date"]].head(10))

=== Excluded participants (num_visits == 1) ===
Total excluded: 2,680

first_date vs last_date — same date? (single visit)
first_date == last_date (true single visit): 2,643
first_date != last_date (different visits!): 37

Sample of excluded with different dates:
                person_id Phecode  num_visits  first_date   last_date
4359    PMBB1014648509276     389         1.0  2017-04-12  2017-04-19
170291  PMBB1544701491150     389         1.0  2021-04-07  2021-04-15
323527  PMBB2037603455996     389         1.0  2019-06-03  2019-06-05
371460  PMBB2183655699089     389         1.0  2017-05-18  2017-06-12
579520  PMBB2829482469486     389         1.0  2022-12-23  2022-12-24
795343  PMBB3491323822576     389         1.0  2015-02-22  2015-03-09
813388  PMBB3553066270742     389         1.0  2018-03-14  2018-03-18
862446  PMBB3712878051491     389         1.0  2018-10-18  2019-02-23
950442  PMBB3993029077789     389         1.0  2023-06-12  2023-06-19
951035  PMBB3994422228884     389   